# Notebook 05: p_censoring

Loads interaction-feature data, creates OOF censoring probability, and saves output.

**Notebook run order**

| Step | Notebook | Purpose | Output |
|---|---|---|---|
| 1 | [01-EDA.ipynb](01-EDA.ipynb) | EDA and baseline profiling | `data/01-EDA.csv` |
| 2 | [02-outlier-cleaning.ipynb](02-outlier-cleaning.ipynb) | Outlier strategy evaluation and cleaning | `data/02-outlier-cleaning.csv` |
| 3 | [03-feature-transformations.ipynb](03-feature-transformations.ipynb) | Per-feature transform scans and apply | `data/03-feature-transformations.csv` |
| 4 | [04-interaction-features.ipynb](04-interaction-features.ipynb) | Interaction scans and feature adds | `data/04-interaction-features.csv` |
| 5 | **[05-p_censoring.ipynb](05-p_censoring.ipynb)** | **OOF censoring probability feature** | **`data/x-final.csv`** |

## 1. Notebook set-up

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

sys.path.insert(0, str(Path.cwd().resolve().parent / 'src'))

from config import DATA_DIR


## 2. Data loading

In [2]:
in_path = DATA_DIR / '04-interaction-features.csv'

if not in_path.exists():
    raise FileNotFoundError(
        (f'Missing required input file: {in_path}. Run 04-interaction-features.ipynb ' + 
         'first to generate data/04-interaction-features.csv.')
    )

housing_df = pd.read_csv(in_path)
print(f'Loaded: {in_path}')

Loaded: data/04-interaction-features.csv


## 3. Censoring model

In [3]:
y_censored = (housing_df['MedHouseVal'] >= 5.0).astype(int)
base_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
clf = LogisticRegression(max_iter=1000, random_state=315)

## 4. Add censoring probability feature

In [4]:
p_censored_oof = cross_val_predict(
    clf,
    housing_df[base_features],
    y_censored,
    cv=10,
    method='predict_proba'
)[:, 1]

housing_df['p_censored'] = p_censored_oof
print('Added p_censored OOF feature.')

Added p_censored OOF feature.


## 5. Save output

In [5]:
out_path = DATA_DIR / 'x-final.csv'
housing_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

Saved: data/x-final.csv
